# Distancias de referencia entre fragmentos

Calcula, sobre las mascaras MHA originales, la distancia 3D de cada fragmento conminuto al fragmento principal de su region (`distancias_fragmentos.measure_fragment_separations`).

- **Entradas:** `splits/dataset.csv` (notebook 01), `evidencias/datos/main_fragments.csv` (notebook 03) y `data_mha/`.
- **Salidas:** `evidencias/distancias_referencia.csv`.

## 1. Configuracion

Rutas relativas a la raiz del repositorio y lectura de los archivos que generan los notebooks anteriores. Las definiciones compartidas se cargan por nombre desde el notebook donde viven (`definiciones_notebooks.cargar_definiciones`), sin ejecutar sus celdas de prueba.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import SimpleITK as sitk

# Raiz del repositorio: funciona si el kernel arranca en notebooks/ o en la raiz.
BASE_DIR = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "requirements.txt").is_file())
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))  # modulos .py del proyecto

SPLITS_DIR = BASE_DIR / "splits"
EVIDENCE_DIR = BASE_DIR / "evidencias"
GENERATED_DATA_DIR = EVIDENCE_DIR / "datos"

def to_repo_relative(path):
    """Guarda rutas portables dentro del repositorio."""
    path = Path(path)
    try:
        return path.relative_to(BASE_DIR).as_posix()
    except ValueError:
        return path.as_posix()


def resolve_repo_path(path):
    """Permite leer rutas relativas o absolutas sin romper compatibilidad."""
    path = Path(path)
    if path.is_absolute():
        return path
    return BASE_DIR / path

# Dataset curado y splits generados por 01_dataset_splits.ipynb
df = pd.read_csv(SPLITS_DIR / "dataset.csv", dtype={"case_id": str})

# Fragmento principal por hueso calculado en 03_eda.ipynb
main_frag_df = pd.read_csv(GENERATED_DATA_DIR / "main_fragments.csv", dtype={"case_id": str})

## 2. Distancia de separación entre fragmentos

Esta sección calcula primero la distancia de referencia usando las máscaras MHA originales, que conservan los IDs de instancia. El fragmento principal de cada región se define como el de mayor volumen, igual que en el EDA. Se usa el espaciado físico del MHA en milímetros y `distance_transform_edt`.

Cada vóxel se interpreta como una celda física con el tamaño indicado por el header MHA. La máscara principal se dilata una celda en los tres ejes y sobre esa representación se calcula el EDT con el spacing anisotrópico. Así se mide la distancia borde a borde entre las celdas voxelizadas: si se tocan por cara, arista o esquina, la distancia es 0 mm. La medida corresponde a las máscaras discretas, no a una superficie anatómica continua. La comparación con predicciones se añadirá cuando el equipo produzca máscaras 3D de instancia.

La tabla final se guarda en `evidencias/distancias_referencia.csv`, con una fila por fragmento y caso. Las pruebas sintéticas de contacto y spacing se ejecutan desde la terminal con `python3 -m unittest discover -s tests -v`.


In [2]:
from distancias_fragmentos import measure_fragment_separations

DISTANCE_CASES = df  # usa df.head(1) para una prueba rápida antes del dataset completo
distance_tables = []
for row in DISTANCE_CASES.itertuples(index=False):
    label_path = resolve_repo_path(row.label_path)
    label_image = sitk.ReadImage(str(label_path))
    instance_volume = sitk.GetArrayFromImage(label_image).astype(np.int16)
    case_distances = measure_fragment_separations(
        instance_volume,
        label_image.GetSpacing(),  # SimpleITK: (x, y, z) mm
        case_id=str(row.case_id),
    )
    distance_tables.append(case_distances)

distancias_referencia_df = pd.concat(distance_tables, ignore_index=True)
distance_table_path = EVIDENCE_DIR / "distancias_referencia.csv"
distancias_referencia_df.to_csv(distance_table_path, index=False)

print("Casos procesados:", distancias_referencia_df["case_id"].nunique())
print("Fragmentos con distancia:", len(distancias_referencia_df))
print("Fragmentos conminutos:", int((~distancias_referencia_df["is_main"]).sum()))
print("Archivo:", to_repo_relative(distance_table_path))
display(distancias_referencia_df.head(12))


Casos procesados: 100
Fragmentos con distancia: 575
Fragmentos conminutos: 275
Archivo: evidencias/distancias_referencia.csv


,case_id,region,fragment_id,main_fragment_id,is_main,n_voxels,distance_mm
0,001,coxal_derecho,21,21,True,844349,0.0
1,001,coxal_izquierdo,11,11,True,762194,0.0
2,001,coxal_izquierdo,12,11,False,63957,0.0
3,001,sacro,1,1,True,458539,0.0
4,002,coxal_derecho,21,21,True,470491,0.0
5,002,coxal_derecho,22,21,False,15260,0.0
6,002,coxal_derecho,23,21,False,19132,0.0
7,002,coxal_izquierdo,11,11,True,502549,0.0
8,002,sacro,1,1,True,323928,0.0
9,002,sacro,2,1,False,39087,0.0


### Interpretación del conteo y de las distancias

Se procesaron **100 casos** y la tabla contiene **575 filas**, una por cada fragmento identificado en las máscaras MHA. De estas filas, **300 corresponden a fragmentos principales** (uno por cada hueso/región presente en cada caso) y **275 a fragmentos conminutos**, es decir, fragmentos adicionales del mismo hueso. Así, “Fragmentos con distancia: 575” es el total de instancias registradas; “Fragmentos conminutos: 275” cuenta solo las instancias cuya separación respecto al fragmento principal se analiza.

El fragmento principal aparece con `distance_mm = 0` por definición: es la referencia y no se mide contra sí mismo. Las distancias pequeñas pueden ocurrir cuando los fragmentos están muy próximos, mientras que valores grandes también pueden ser válidos si están más separados.

Para interpretar cualquier valor, se debe comprobar que ambos fragmentos estén correctamente asociados al mismo hueso y que el cálculo use el espaciado físico correcto del MHA. Una distancia grande no es automáticamente un error, pero sí conviene revisar visualmente los casos extremos y verificar la máscara y la pertenencia anatómica.

La medición se calcula en 3D sobre máscaras voxelizadas, interpretando cada vóxel como una celda física; no es una distancia entre superficies anatómicas continuas o mallas. Las capturas son proyecciones 2D para inspección visual, por lo que no representan por sí solas la distancia 3D.

## 3. Revisión de la tabla de distancias

Se resumen las distancias de los fragmentos conminutos y se inspeccionan los valores más grandes. También se comprueba que el fragmento principal seleccionado por el cálculo coincida con el EDA previo. Los valores extremos deben revisarse sobre la imagen y la máscara antes de interpretarlos.


In [3]:
conminuted_distances_df = distancias_referencia_df[~distancias_referencia_df["is_main"]].copy()
print("Resumen de distancia a fragmento principal (mm):")
display(conminuted_distances_df["distance_mm"].describe().to_frame().T)
print("Resumen por región anatómica (mm):")
display(conminuted_distances_df.groupby("region")["distance_mm"].agg(["count", "median", "max"]))
print("Mayores distancias: revisar estos casos visualmente")
display(conminuted_distances_df.nlargest(10, "distance_mm")[
    ["case_id", "region", "fragment_id", "main_fragment_id", "distance_mm"]
])

principal_check = (
    distancias_referencia_df[distancias_referencia_df["is_main"]]
    .merge(
        main_frag_df[["case_id", "region", "main_label"]],
        left_on=["case_id", "region", "fragment_id"],
        right_on=["case_id", "region", "main_label"],
        how="outer",
        indicator=True,
    )
)
assert (principal_check["_merge"] == "both").all()
print("Fragmentos principales coinciden con el EDA:", len(principal_check), "/", len(main_frag_df))


Resumen de distancia a fragmento principal (mm):


,count,mean,std,min,25%,50%,75%,max
distance_mm,275.0,1.28512,6.512351,0.0,0.0,0.0,0.0,71.811198


Resumen por región anatómica (mm):


,count,median,max
region,,,
coxal_derecho,107,0.0,71.811198
coxal_izquierdo,115,0.0,49.369645
sacro,53,0.0,10.056456


Mayores distancias: revisar estos casos visualmente


,case_id,region,fragment_id,main_fragment_id,distance_mm
547,096,coxal_derecho,23,21,71.811198
484,085,coxal_izquierdo,12,11,49.369645
309,054,coxal_derecho,24,21,39.391438
218,039,coxal_derecho,23,21,33.355780
24,005,coxal_derecho,23,21,23.325827
130,024,coxal_izquierdo,13,11,19.078536
565,099,coxal_izquierdo,12,11,12.880002
308,054,coxal_derecho,23,21,12.577351
478,084,coxal_izquierdo,14,11,11.562614
209,037,sacro,2,1,10.056456


Fragmentos principales coinciden con el EDA: 300 / 300
